# Tratamento das Bases semanais do RH

**Inputs**\
Planilhas semanais enviadas pelo time de RH com dados dos colaboradores desligados

**Outputs**\
Planilha unica com os desligados dos ultimos 24 meses + ativos


In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

In [0]:
# Parametro do nome da tabela da competência atual. Ex: 202404
dbutils.widgets.text("nmmes", "")

# Parametro nome do arquivo enviado pelo RH com os desligados. Ex: 20240423
dbutils.widgets.text("dtdesligados", "")

In [0]:
dtdesligados = dbutils.widgets.get("dtdesligados")
nmmes = dbutils.widgets.get("nmmes")

caminho_desligados = f"/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_rh/base_desligados/RH_BASE_DESLIGADOS_{dtdesligados}.xlsx" 

caminho_rh_previa = f"/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_rh/previa/"

In [0]:
df_desligados = read_excel(caminho_desligados,"'BASE_DESLIGADOS'!A1:AD1048576")

In [0]:
import os
from pyspark.sql import DataFrame

caminho = "/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_rh/previa/"

files = dbutils.fs.ls(caminho)

dataframes = {}

for f in files:
    if "PREVIA_SEMANAL_RH_" in f.name:
        nome_df = os.path.splitext(f.name)[0]  # nome do arquivo sem extensão
        caminho_arquivo = f.path

        df = spark.read \
            .format("com.crealytics.spark.excel") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(caminho_arquivo)

        dataframes[nome_df] = df


df_previa_rh_final: DataFrame = None

for df in dataframes.values():
    if df_previa_rh_final is None:
        df_previa_rh_final = df
    else:
        df_previa_rh_final = df_previa_rh_final.unionByName(df, allowMissingColumns=True)

colunas_validas = ["MATRICULA", "STATUS_Colaborador", "DATA_ADMISSAO","DATA_DESLIGAMENTO"]  # substitua pelas colunas desejadas
df_previa_rh_final = df_previa_rh_final.dropDuplicates(colunas_validas)

df_previa_rh_final = df_previa_rh_final.toDF(*[col.upper() for col in df_previa_rh_final.columns])

df_previa_rh_final = df_previa_rh_final.drop("EMAIL")

df_previa_rh_final.createOrReplaceTempView('df_previa_rh_final')  

In [0]:
print(df_previa_rh_final.columns)

#### Base de desligados historica abaixo

In [0]:
print(df_desligados.columns)

In [0]:
from pyspark.sql.functions import *

try:
    # Um dicionário mapeando nome da coluna para o novo tipo
    novos_tipos = {
        "CodEmpresa": "integer",
        "MATRICULA": "bigint",
        "CPF": "bigint",
        "Nascimento": "date",
        "Admissao": "date",
        "Desligamento": "date"
    }

    # Aplicando os casts
    for coluna, tipo in novos_tipos.items():
        df_desligados = df_desligados.withColumn(coluna, col(coluna).cast(tipo))
    
    # Formatando as datas para remover a hora
    colunas_data = ["Nascimento", "Admissao", "Desligamento"]
    for coluna in colunas_data:
        df_desligados = df_desligados.withColumn(coluna, date_format(col(coluna), "yyyy-MM-dd"))



    df_desligados = df_desligados.toDF(*[col.upper() for col in df_desligados.columns])

    renomear_colunas = {
        "CODEMPRESA": "COD_EMPRESA",
        "NASCIMENTO": "DATA_NASCIMENTO",
        "STATUS": "STATUS_COLABORADOR",
        "ADMISSAO": "DATA_ADMISSAO",
        "TEMPOEMPRESA": "TEMPO_EMPRESA",
        "FUNCAO": "DESC_FUNCAO_ATUAL",
        "DESLIGAMENTO": "DATA_DESLIGAMENTO",
        "TIPODESLIGAMENTO": "TIPO_DESLIGAMENTO",
        "TIPODEFICIENCIA": "DESC_TIPO_DEFICIENCIA",
        "AGRUPAMENTOCARGO": "AGRUP_FUNCAO",
        "MATRICULAGESTOR": "MATRICULA_GESTOR",
        "NOMEGESTOR": "NOME_GESTOR",
        "CODCRESULTADO": "COD_CENTRO_RESULTADO",
        "CODCCUSTO": "COD_CENTRO_CUSTO_FORM",
        "DESCRICAON1":"NIVEL1",
        "DESCRICAON2":"NIVEL2",
        "DESCRICAON3":"NIVEL3",
        "DESCRICAON4":"NIVEL4",
        "DESCRICAON5":"NIVEL5",
        "ESTABELECIMENTO":"COD_ESTABELECIMENTO",
    }

    # Aplicar os renomes
    for antigo, novo in renomear_colunas.items():
        df_desligados = df_desligados.withColumnRenamed(antigo, novo)
    
    # NOVA PARTE: Formatando as datas para remover a hora (depois de renomear)
    colunas_data_formatar = ["DATA_NASCIMENTO", "DATA_ADMISSAO", "DATA_DESLIGAMENTO"]
    for coluna in colunas_data_formatar:
        if coluna in df_desligados.columns:
            df_desligados = df_desligados.withColumn(coluna, date_format(col(coluna), "yyyy-MM-dd"))



    df_desligados.createOrReplaceTempView('df_desligados')

except Exception as e:
    print(e)

In [0]:
# Define the common columns
common_cols = [col.upper() for col in df_desligados.columns if col.upper() in [c.upper() for c in df_previa_rh_final.columns]]

# Select only the common columns from both DataFrames
df_desligados_sel = df_desligados.select(*common_cols)
df_previa_rh_final_sel = df_previa_rh_final.select(*common_cols)

# Register temp views
df_desligados_sel.createOrReplaceTempView('df_desligados_sel')
df_previa_rh_final_sel.createOrReplaceTempView('df_previa_rh_final_sel')

# Perform UNION
df_rh_hist_com_previa = spark.sql('''
    SELECT * FROM df_desligados_sel
    UNION
    SELECT * FROM df_previa_rh_final_sel
''')

df_rh_hist_com_previa = df_rh_hist_com_previa.withColumn(
    "STATUS_COLABORADOR", 
    upper(df_rh_hist_com_previa["STATUS_COLABORADOR"])
)

df_rh_hist_com_previa = df_rh_hist_com_previa.replace(
    "DESLIGADO-R", 
    "DESLIGADO", 
    subset=["STATUS_COLABORADOR"]
)

colunas_validas = [
    "MATRICULA", 
    "STATUS_COLABORADOR", 
    "DATA_ADMISSAO",
    "DATA_DESLIGAMENTO"
]
df_rh_hist_com_previa = df_rh_hist_com_previa.dropDuplicates(colunas_validas)

df_rh_hist_com_previa = df_rh_hist_com_previa.withColumn("MES_ADMISSAO", trunc("DATA_ADMISSAO", "MM"))

df_rh_hist_com_previa = df_rh_hist_com_previa.withColumn("MES_DESLIGAMENTO", trunc("DATA_DESLIGAMENTO", "MM"))

df_rh_hist_com_previa.createOrReplaceTempView('df_rh_hist_com_previa')

In [0]:
print(df_previa_rh_final.columns)

In [0]:
print(df_desligados.columns)

In [0]:
df_rh_hist_com_previa_f = spark.sql('''
    SELECT A.* FROM df_rh_hist_com_previa A LEFT JOIN(
    SELECT MATRICULA, MAX(MES_DESLIGAMENTO) AS ULTIMO_DESLIGAMENTO FROM df_rh_hist_com_previa
    GROUP BY ALL) AS B
    ON A.MATRICULA = B.MATRICULA AND A.MES_DESLIGAMENTO = B.ULTIMO_DESLIGAMENTO
''')

df_rh_hist_com_previa_f.createOrReplaceTempView('df_rh_hist_com_previa_f')

In [0]:
from pyspark.sql.functions import col, regexp_replace, lpad, when

# 1. Carrega a 'view' da célula anterior
df = spark.table('df_rh_hist_com_previa_f')

# 2. Converte o CPF para string (é aqui que a notação científica aparece)
df = df.withColumn("CPF_STR", col("CPF").cast("string"))

# 3. Limpa TUDO que não for número (pontos, traços, e o ".0" ou "E10" da notação científica)
df = df.withColumn("CPF_CLEAN", regexp_replace(col("CPF_STR"), r"[^0-9]", ""))

# 4. Cria a coluna 'CPFN' final, garantindo que ela tenha 11 dígitos, com zeros à esquerda
#    Esta é a lógica de limpeza e padding que faltava
df = df.withColumn("CPFN", lpad(col("CPF_CLEAN"), 11, "0"))

# 5. Salva o resultado em 'desligados_1' e cria a 'view' para a Célula 19 usar
desligados_1 = df
desligados_1.createOrReplaceTempView('desligados_1')

In [0]:
#desligados_1 = spark.sql('''
 #   SELECT *,
  #  CASE WHEN LENGTH(CAST(CPF AS STRING)) == 10 THEN CONCAT("0", CAST(CPF AS STRING))
   #         ELSE CAST(CPF AS STRING) END AS CPFN
    #FROM df_rh_hist_com_previa_f
#''')

#desligados_1.createOrReplaceTempView('desligados_1')

In [0]:
df_fecham_financ = spark.sql(f'''SELECT * FROM databox.juridico_comum.tb_fecham_financ_trab_{nmmes}''')

from pyspark.sql.functions import regexp_replace

# Substituir espaços da coluna "nome_coluna"
df_fecham_financ = df_fecham_financ.withColumn("MATRICULA", regexp_replace("MATRICULA", " ", ""))

df_fecham_financ.createOrReplaceTempView('df_fecham_financ')

In [0]:
%sql
SELECT * FROM df_fecham_financ LIMIT 10

In [0]:
df_fecham_financ_cons_1 = spark.sql('''
    select A.ID_PROCESSO, coalesce(A.MATRICULA, B.MATRICULA) AS MATRICULA, cast(coalesce(A.MATRICULA,B.MATRICULA) as int) as MATRICULAINT, coalesce(A.PARTE_CONTRARIA_CPF, B.PARTE_CONTRARIA_CPF) AS CPF, A.MES_FECH, A.NOVOS from df_fecham_financ A
    LEFT JOIN databox.juridico_comum.trab_ger_consolida_20250623 B
    ON A.ID_PROCESSO = B.PROCESSO_ID
''') 

from pyspark.sql.functions import regexp_replace

# Substituir espaços da coluna "nome_coluna"
df_fecham_financ_cons_1 = df_fecham_financ_cons_1.withColumn("CPF", regexp_replace("CPF",  r"[.,-]", ""))

df_fecham_financ_cons_1.createOrReplaceTempView('df_fecham_financ_cons_1')

In [0]:
df_fecham_financ_cons_2 = spark.sql('''
    SELECT *
    ,CASE WHEN LENGTH(CPF) == 11 THEN CONCAT(SUBSTRING(CPF,1,3),'.',SUBSTRING(CPF,4,3),'.',SUBSTRING(CPF,7,3),'-',SUBSTRING(CPF,10,2))
        ELSE CPF END AS CPFN

    FROM df_fecham_financ_cons_1
''')

from pyspark.sql.functions import regexp_replace

# Remover todos os espaços (um ou mais) da coluna "nome"
df_fecham_financ_cons_2 = df_fecham_financ_cons_2.withColumn("MATRICULA", regexp_replace("MATRICULA", r"\s+", ""))

df_fecham_financ_cons_2.createOrReplaceTempView('df_fecham_financ_cons_2')

In [0]:
%sql
SELECT 
  CPF,    -- Esta é a coluna APÓS a limpeza de pontos/traços (da Célula 17)
  CPFN    -- Esta é a coluna APÓS a re-formatação (da Célula 18)
FROM df_fecham_financ_cons_2 
LIMIT 20

In [0]:
desligados_2 = spark.sql('''
    select * 
    ,CASE WHEN LENGTH(CPFN) == 11 THEN CONCAT(SUBSTRING(CPFN,1,3),'.',SUBSTRING(CPFN,4,3),'.',SUBSTRING(CPFN,7,3),'-',SUBSTRING(CPFN,10,2))
            ELSE CPFN END AS CPFN2
    from desligados_1
''')

desligados_2 = desligados_2.withColumn("MATRICULA", lpad("MATRICULA", 8, "0"))

desligados_2.createOrReplaceTempView('desligados_2')

In [0]:
%sql
SELECT 
  CPF,    -- Este é o CPF "sujo" original da base de RH
  CPFN,   -- Este é o resultado da Célula 15
  CPFN2   -- Este é o resultado final da Célula 19 (que vai no JOIN)
FROM desligados_2
LIMIT 20

In [0]:
# %sql

# select A.MATRICULA, B.MATRICULA, A.CPFN2, B.CPFN, B.ID_PROCESSO, B.MES_FECH
# FROM desligados_2 A
# LEFT JOIN df_fecham_financ_cons_2 B 
# ON A.CPFN2 = B.CPFN

In [0]:
df_desligados_com_processos = spark.sql('''
    SELECT 
    A.COD_EMPRESA
    ,A.MATRICULA
    ,A.NOME
    ,A.CPFN AS CPF
    ,A.ESCOLARIDADE
    ,A.GENERO
    ,A.DATA_NASCIMENTO as DATA_NASCIMENTO
    ,A.STATUS_COLABORADOR as STATUS
    ,A.DATA_ADMISSAO as DATA_ADMISSAO
    ,A.TEMPO_EMPRESA as TEMPO_EMPRESA
    ,A.DESC_FUNCAO_ATUAL as CARGO
    ,A.DATA_DESLIGAMENTO as DATA_DESLIGAMENTO
    ,A.TIPO_DESLIGAMENTO as TIPO_DESLIGAMENTO
    ,A.MOTIVO_DESLIGAMENTO
    ,A.AGRUP_FUNCAO as AGRUPAMENTO_CARGO
    ,A.MATRICULA_GESTOR as MATRICULA_GESTOR
    ,A.NOME_GESTOR as NOME_GESTOR
    ,A.COD_CENTRO_RESULTADO as COD_CENTRO_CUSTO
    ,A.ESTRUTURA as ESTRUTURA
    ,A.NIVEL1 AS DESCRICAON1
    ,A.NIVEL2 AS DESCRICAON2
    ,A.NIVEL3 AS DESCRICAON3
    ,A.NIVEL4 AS DESCRICAON4
    ,A.COD_ESTABELECIMENTO AS ESTABELECIMENTO
    ,B.ID_PROCESSO
    ,B.MES_FECH
    ,coalesce(B.NOVOS, 0) AS NOVOS
    FROM desligados_2 A
    LEFT JOIN df_fecham_financ_cons_2 B
    ON A.CPFN2 = B.CPFN
''')

df_desligados_com_processos.createOrReplaceTempView('df_desligados_com_processos')

In [0]:
df_desligados_com_processos_f = spark.sql('''
    SELECT * 
        ,TRUNC(DATA_DESLIGAMENTO, 'MM') AS MES_DESLIGAMENTO
        ,ROW_NUMBER() OVER (PARTITION BY MATRICULA ORDER BY MES_FECH) AS RN
    FROM df_desligados_com_processos
''')

df_desligados_com_processos_f.createOrReplaceTempView('df_desligados_com_processos_f')



In [0]:
df_desligados_com_processos_ff = spark.sql('''
    SELECT *,
    CASE WHEN RN = 1 THEN 1
    ELSE 0 END AS QTDE

    FROM df_desligados_com_processos_f
''')


df_desligados_com_processos_ff.createOrReplaceTempView('df_desligados_com_processos_ff')

In [0]:
df_desligados_com_processos_ff.head()

In [0]:
%sql

SELECT STATUS, sum(novos) as TOTAL FROM df_desligados_com_processos_f
group by all

In [0]:
import pandas as pd
from shutil import copyfile
import xlsxwriter
# Convert PySpark DataFrame to Pandas DataFrame
pandas_df = df_desligados_com_processos_ff.toPandas()

# Save the Pandas DataFrame to an Excel file
local_path = f'/local_disk0/tmp/TB_DESLIGADOS.xlsx'
pandas_df.to_excel(local_path, index=False, sheet_name='DESLIGADOS_RH_F', engine='xlsxwriter')

# Copy the file from the local disk to the desired volume
volume_path = f'/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_indicadores_resultado/TB_DESLIGADOS.xlsx'

copyfile(local_path, volume_path)

In [0]:
%sql
SELECT * FROM databox.default.df_fecham_financ_cons_2